# PRISM - CLIP + SPIDER-Breast
# Pathology Reliability In Scarce-label Medicine

In [1]:
import torch
import numpy as np
import pandas as pd
import os
from google.colab import userdata, drive
from torchvision import transforms

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/clip/spider_breast'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


In [2]:
!pip install transformers timm einops huggingface_hub scikit-learn scipy tqdm -q

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Login OK")

Login OK


In [3]:
from transformers import CLIPModel, CLIPProcessor
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model = model.cuda().eval()
print("CLIP loaded!")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded!


In [4]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [8]:
import json
import time
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image

class SPIDERBreastDataset(Dataset):
    def __init__(self, records, img_dir, transform=None):
        self.records = records
        self.img_dir = img_dir
        self.transform = transform
        classes = sorted(set(r['class'] for r in records))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.classes = classes

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        img_path = f'{self.img_dir}/{r["image_name"]}'
        for attempt in range(5):
            try:
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                label = self.class_to_idx[r['class']]
                return img, label
            except OSError:
                if attempt < 4:
                    time.sleep(2 ** attempt)
                else:
                    img = Image.new('RGB', (224, 224), 0)
                    if self.transform:
                        img = self.transform(img)
                    return img, self.class_to_idx[r['class']]

SPIDER_DIR = '/content/drive/MyDrive/PRISM/datasets/spider_breast/extracted/SPIDER-breast'
IMG_DIR    = f'{SPIDER_DIR}/images'
META_PATH  = f'{SPIDER_DIR}/metadata.json'

with open(META_PATH) as f:
    meta = json.load(f)

train_records = [r for r in meta if r['split'] == 'train']
test_records  = [r for r in meta if r['split'] == 'test']

val_size   = int(0.15 * len(train_records))
import random
random.seed(42)
random.shuffle(train_records)
val_records   = train_records[:val_size]
train_records = train_records[val_size:]

train_dataset = SPIDERBreastDataset(train_records, IMG_DIR, transform)
val_dataset   = SPIDERBreastDataset(val_records,   IMG_DIR, transform)
test_dataset  = SPIDERBreastDataset(test_records,  IMG_DIR, transform)

print(f"Classes ({len(train_dataset.classes)}):")
for i, c in enumerate(train_dataset.classes):
    print(f"  {i}: {c}")

print(f"\nTrain: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

Classes (18):
  0: Adenosis
  1: Benign phyllodes tumor
  2: Ductal carcinoma in situ (high-grade)
  3: Ductal carcinoma in situ (low-grade)
  4: Fat
  5: Fibroadenoma
  6: Fibrocystic changes
  7: Fibrosis
  8: Invasive non-special type carcinoma
  9: Lipogranuloma
  10: Lobular invasive carcinoma
  11: Malignant phyllodes tumor
  12: Necrosis
  13: Normal ducts
  14: Normal lobules
  15: Sclerosing adenosis
  16: Typical ductal hyperplasia
  17: Vessels

Train: 68,730
Val:   12,128
Test:  12,034


In [ ]:
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(device)
            outputs = model.vision_model(pixel_values=images)
            features = outputs.pooler_output
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy() if hasattr(labels, 'numpy') else np.array(labels))
    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting: 100%|██████████| 269/269 [17:41:26<00:00, 236.75s/it]


Train: (68730, 768)
Extracting val features...


Extracting: 100%|██████████| 48/48 [2:55:05<00:00, 218.87s/it]


Val: (12128, 768)
Extracting test features...


Extracting: 100%|██████████| 48/48 [2:53:34<00:00, 216.97s/it]

Test: (12034, 768)


In [ ]:
np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy',   train_labels)
np.save(f'{EMBED_DIR}/val_features.npy',   val_features)
np.save(f'{EMBED_DIR}/val_labels.npy',     val_labels)
np.save(f'{EMBED_DIR}/test_features.npy',  test_features)
np.save(f'{EMBED_DIR}/test_labels.npy',    test_labels)
print(f"Embeddings saved to: {EMBED_DIR}")

Embeddings saved to: /content/drive/MyDrive/PRISM/embeddings/clip/spider_breast


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def compute_ece_multiclass(y_true, y_prob, n_bins=15):
    confidence  = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    correct     = (predictions == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(correct[mask].mean() - confidence[mask].mean())
    return ece / len(y_true)

def stratified_sample(train_l, n, seed):
    np.random.seed(seed)
    classes = np.unique(train_l)
    idx = []
    for c in classes:
        c_idx = np.where(train_l == c)[0]
        idx.append(np.random.choice(c_idx, 1)[0])
    already = set(idx)
    remaining = n - len(idx)
    if remaining > 0:
        pool = [i for i in range(len(train_l)) if i not in already]
        extra = np.random.choice(pool, min(remaining, len(pool)), replace=False)
        idx.extend(extra.tolist())
    return np.array(idx[:n])

def run_label_fraction(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    n_classes = len(np.unique(train_l))
    n = max(int(len(train_l) * fraction), n_classes)
    idx = stratified_sample(train_l, n, seed)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    y_prob = clf.predict_proba(test_f)
    y_pred = clf.predict(test_f)
    if y_prob.shape[1] < n_classes:
        full_prob = np.zeros((len(test_f), n_classes))
        for i, c in enumerate(clf.classes_):
            full_prob[:, c] = y_prob[:, i]
        y_prob = full_prob
    return {
        'model': 'CLIP', 'dataset': 'SPIDER-Breast',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_l, y_prob, multi_class='ovr', average='macro', labels=list(range(n_classes))),
        'f1':    f1_score(test_l, y_pred, average='macro', zero_division=0),
        'ece':   compute_ece_multiclass(test_l, y_prob),
        'brier': np.mean([brier_score_loss((test_l==c).astype(int), y_prob[:,c]) for c in range(n_classes)]),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.8897 | ECE: 0.1666 | Brier: 0.0453
  1% | seed 123 | AUROC: 0.8930 | ECE: 0.1633 | Brier: 0.0457
  1% | seed 456 | AUROC: 0.8962 | ECE: 0.1316 | Brier: 0.0461
  5% | seed 42 | AUROC: 0.9264 | ECE: 0.2498 | Brier: 0.0376
  5% | seed 123 | AUROC: 0.9243 | ECE: 0.2634 | Brier: 0.0379
  5% | seed 456 | AUROC: 0.9213 | ECE: 0.2457 | Brier: 0.0379
  10% | seed 42 | AUROC: 0.9364 | ECE: 0.2340 | Brier: 0.0336
  10% | seed 123 | AUROC: 0.9363 | ECE: 0.2324 | Brier: 0.0335
  10% | seed 456 | AUROC: 0.9337 | ECE: 0.2275 | Brier: 0.0341
  25% | seed 42 | AUROC: 0.9487 | ECE: 0.1933 | Brier: 0.0290
  25% | seed 123 | AUROC: 0.9488 | ECE: 0.1862 | Brier: 0.0291
  25% | seed 456 | AUROC: 0.9482 | ECE: 0.1853 | Brier: 0.0292
  50% | seed 42 | AUROC: 0.9570 | ECE: 0.1560 | Brier: 0.0262
  50% | seed 123 | AUROC: 0.9573 | ECE: 0.1565 | Brier: 0.0261
  50% | seed 456 | AUROC: 0.9568 | ECE: 0.1554 | Brier: 0.0261
  100% | seed 42 | AUROC: 0.9641 | ECE: 0.1267 | Brier: 0.0236
  1

In [ ]:
from scipy.optimize import minimize_scalar

def find_temperature_multiclass(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    n_classes = len(np.unique(train_l))
    n = max(int(len(train_l) * fraction), n_classes)
    idx = stratified_sample(train_l, n, seed)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    val_logits = clf.decision_function(val_f)
    if val_logits.ndim == 1:
        val_logits = np.vstack([-val_logits, val_logits]).T
    T = find_temperature_multiclass(val_logits, val_l)
    test_prob_raw = clf.predict_proba(test_f)
    if test_prob_raw.shape[1] < n_classes:
        full_prob = np.zeros((len(test_f), n_classes))
        for i, c in enumerate(clf.classes_):
            full_prob[:, c] = test_prob_raw[:, i]
        test_prob_raw = full_prob
    ece_raw     = compute_ece_multiclass(test_l, test_prob_raw)
    test_logits = clf.decision_function(test_f)
    if test_logits.ndim == 1:
        test_logits = np.vstack([-test_logits, test_logits]).T
    scaled      = test_logits / T
    exp_s       = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s = exp_s / exp_s.sum(axis=1, keepdims=True)
    if test_prob_s.shape[1] < n_classes:
        full_prob_s = np.zeros((len(test_f), n_classes))
        for i, c in enumerate(clf.classes_):
            full_prob_s[:, c] = test_prob_s[:, i]
        test_prob_s = full_prob_s
    ece_scaled  = compute_ece_multiclass(test_l, test_prob_s)
    return {
        'model': 'CLIP', 'dataset': 'SPIDER-Breast',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc':       roc_auc_score(test_l, test_prob_raw, multi_class='ovr', average='macro', labels=list(range(n_classes))),
        'brier_raw':   np.mean([brier_score_loss((test_l==c).astype(int), test_prob_raw[:,c]) for c in range(n_classes)]),
        'brier_scaled':np.mean([brier_score_loss((test_l==c).astype(int), test_prob_s[:,c])   for c in range(n_classes)]),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           0.3564   0.1538      0.0727           0.0812
0.05           0.4134   0.2530      0.0229           0.2301
0.10           0.4688   0.2313      0.0231           0.2082
0.25           0.5292   0.1883      0.0262           0.1621
0.50           0.5680   0.1560      0.0307           0.1253
1.00           0.6065   0.1262      0.0294           0.0969


In [ ]:
df.to_csv(f'{RESULTS_DIR}/clip_spider_breast_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/clip_spider_breast_temperature_scaling.csv', index=False)
print("Results saved!")
print(f"  {RESULTS_DIR}/clip_spider_breast_results.csv")
print(f"  {RESULTS_DIR}/clip_spider_breast_temperature_scaling.csv")

Results saved!
  /content/drive/MyDrive/PRISM/results/clip_spider_breast_results.csv
  /content/drive/MyDrive/PRISM/results/clip_spider_breast_temperature_scaling.csv
